# PHEME Four Behavioral Signatures Experiment

原始单文件脚本被拆分为：

- `pheme_experiment_utils.py`：全部工具函数、数据加载、Prompt、实验运行与分析函数。
- `pheme_four_effects.ipynb`：实验配置、数据初始化以及四个独立实验。

四个实验各自位于一个独立代码单元中：

1. Threshold / Binary Activation
2. Anchoring
3. Format Sensitivity
4. Label Override

ottawashooting thread id:
"524947030616313856"
"524947716393414656"
"525046443103354880"
"525051210349289472"
"524930851747164160"

ferguson thread id:
"500293392060780546"
"500361302238564352"
"500335355904540674"
"500307001629745152"
"500298847550472194"

sydneysiege thread id:
"544349042952916993"
"544304742743044096"
"544329935943237632"
"544445364322197504"
"544454229960974336"




## 1. 导入工具模块与配置参数

In [37]:
from pathlib import Path
from collections import Counter
from datetime import datetime
import os
import sys
import time

CURRENT_DIR = Path.cwd().resolve()
if (
    (CURRENT_DIR / "Pheme" / "pheme_four_effects.ipynb").exists()
    and (CURRENT_DIR / "Pheme" / "experiment_util.py").exists()
):
    REPO_ROOT = CURRENT_DIR / "Pheme"
else:
    REPO_ROOT = CURRENT_DIR
if not (REPO_ROOT / "experiment_util.py").exists():
    raise FileNotFoundError(f"Expected experiment_util.py at {REPO_ROOT}")
WORKSPACE_ROOT = REPO_ROOT.parent
DATA_ROOT = REPO_ROOT / "data"
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Expected PHEME data folder at {DATA_ROOT}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import experiment_util as exp


# =========================
# Data configuration
# =========================
EVENT_NAME = "sydneysiege"
EVENT_DIR = DATA_ROOT / EVENT_NAME
THREAD_ID = "544304742743044096"
MIN_REACTIONS = 10
MAX_AGENTS = 200
OUTPUT_DIR = WORKSPACE_ROOT / "pheme_four_effects_results"
LLM_OUTPUT_DIR = WORKSPACE_ROOT / "output"

# =========================
# Model configuration
# =========================
API_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions"
API_KEY = "sk-ws-H.ELRIHIL.4UZk.MEUCIQDu7iCV-gHct4Z-QTh1-kV0_m6OXVr3JkgKGOLUJI0m_QIgS0mPBO0ZdXPH9qF8yijNUVaPTQKq51KBI4tF_VUEuT8"
MODEL = "qwen3.6-flash"

# =========================
# Runtime configuration
# =========================
TEMPERATURE = 0.7
REPETITIONS = 3
MAX_WORKERS = 5
SLEEP_BETWEEN = 0.2
MAX_NEIGHBORS_IN_PROMPT = 8
SEED = 42
TARGET_AGENTS = 30
THRESHOLD_SAMPLE_AGENTS = 20

# =========================
# Experiment controls
# =========================
THRESHOLD_TARGET_STANCES = ["support", "oppose"]
THRESHOLD_COUNTS = [0, 1,2, 3,4, 5, 6]
THRESHOLD_TOTAL_NEIGHBORS = 6
THRESHOLD_SUPPORT_NEIGHBORS = THRESHOLD_TOTAL_NEIGHBORS

ANCHOR_TARGET_STANCES = ["support", "oppose"]
ANCHOR_SUPPORT_NEIGHBORS = 2
ANCHOR_OPPOSITE_NEIGHBORS = 2

FORMAT_TARGET_STANCES = ["oppose"]
FORMAT_SUPPORTS = 3
FORMAT_DENIES = 0

ROLE_FRAC = 0.2

if not API_KEY:
    print("Warning: DASHSCOPE_API_KEY is empty. Set DASHSCOPE_API_KEY before executing API cells.")

exp.configure_runtime(
    api_url=API_URL,
    api_key=API_KEY,
    model=MODEL,
    max_workers=MAX_WORKERS,
    sleep_between=SLEEP_BETWEEN,
    max_neighbors_in_prompt=MAX_NEIGHBORS_IN_PROMPT,
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LLM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. 加载 PHEME thread 并初始化实验状态

In [38]:
(
    selected_thread,
    GRAPH,
    AGENT_NAMES,
    AGENT_TEXTS,
    INITIAL_OPINIONS,
    TOPIC,
) = exp.load_and_set_pheme_thread(
    EVENT_DIR,
    thread_id=THREAD_ID,
    min_reactions=MIN_REACTIONS,
    max_agents=MAX_AGENTS,
)

hub_nodes, peripheral_nodes, degrees = exp.build_role_groups(
    GRAPH,
    frac=ROLE_FRAC,
)

initial_distribution = dict(
    sorted(
        Counter(
            INITIAL_OPINIONS.values()
        ).items()
    )
)

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

all_results = {}
all_analyses = {}
pairwise = {}

print(f"Selected thread: {selected_thread}")
print(f"Topic: {TOPIC[:160]}")
print(f"Model: {MODEL}")
print(
    "Init model:",
    "data_stance",
)
print(f"Repetitions: {REPETITIONS}")

exp.summarize_graph(
    GRAPH,
    degrees,
)

print(
    "Initial stance distribution:",
    initial_distribution,
)

[PHEME] Selected cleaned thread: C:\Users\LiMingzhe\Desktop\LLM Simulation\test\Pheme\data\sydneysiege\544304742743044096_cleaned.json
[PHEME] Event folder: sydneysiege
[PHEME] Comments: 30
[PHEME] Loaded cleaned JSON: C:\Users\LiMingzhe\Desktop\LLM Simulation\test\Pheme\data\sydneysiege\544304742743044096_cleaned.json
[PHEME] Built edges from cleaned reply metadata: 30
[Init] Agent 1/31
[Init Opinion] LLM not called | reason=cleaned_stance | stance=support | agent=0
[Init] Agent 2/31
[Init Opinion] LLM not called | reason=cleaned_stance | stance=support | agent=1
[Init] Agent 3/31
[Init Opinion] LLM not called | reason=cleaned_stance | stance=support | agent=2
[Init] Agent 4/31
[Init Opinion] LLM not called | reason=cleaned_stance | stance=oppose | agent=3
[Init] Agent 5/31
[Init Opinion] LLM not called | reason=cleaned_stance | stance=support | agent=4
[Init] Agent 6/31
[Init Opinion] LLM not called | reason=cleaned_stance | stance=support | agent=5
[Init] Agent 7/31
[Init Opinion] L

## 3. Experiment 1 — Threshold / Binary Activation

同一批 Agent 分别看到 0、1、3、5、7、10 个反对型邻居，总邻居数固定为 10，比较意见变化率、向反对方向移动率和平均变化幅度。


In [39]:
# ============================================================
# Experiment 1:
# Threshold effect on the real comment-reply graph
# ============================================================
import experiment_util as exp

required_neighbor_count = max(
    THRESHOLD_COUNTS
)

# source tweet ????? 0?
# source ????????????? Agent?
target_comment_agents = sorted(
    agent_id
    for agent_id, stance in INITIAL_OPINIONS.items()
    if (
        agent_id != 0
        and exp.normalize_stance(stance) in THRESHOLD_TARGET_STANCES
    )
)

(
    threshold_neighbor_rankings,
    dropped_threshold_agents,
) = exp.build_threshold_neighbor_rankings(
    GRAPH,
    INITIAL_OPINIONS,
    target_agent_ids=target_comment_agents,
    required_count=required_neighbor_count,
    required_support_count=THRESHOLD_SUPPORT_NEIGHBORS,
    seed=SEED,
)

# ?? repetition ????? Agent ???? 20 ??????????? agent ???
threshold_candidate_agents = sorted(
    threshold_neighbor_rankings
)

sampled_threshold_agents_by_rep = exp.sample_threshold_agents_by_repetition(
    threshold_candidate_agents,
    sample_size=THRESHOLD_SAMPLE_AGENTS,
    repetitions=REPETITIONS,
    seed=SEED,
)

threshold_agents = sorted({
    agent_id
    for rep_agents in sampled_threshold_agents_by_rep.values()
    for agent_id in rep_agents
})

sampled_threshold_agents_for_json = {
    str(rep + 1): agent_ids
    for rep, agent_ids in sorted(
        sampled_threshold_agents_by_rep.items()
    )
}

print(
    f"[Threshold] Candidate agents: "
    f"{len(target_comment_agents)}"
)

print(
    f"[Threshold] Valid agents: "
    f"{len(threshold_candidate_agents)}"
)

print(
    f"[Threshold] Sampled unique agents: "
    f"{len(threshold_agents)}"
)

print(
    f"[Threshold] Agents per repetition: "
    f"{THRESHOLD_SAMPLE_AGENTS}"
)

print(
    f"[Threshold] Dropped agents: "
    f"{len(dropped_threshold_agents)}"
)

if not threshold_candidate_agents:
    raise RuntimeError(
        "No comment agent has enough eligible "
        f"opposing comments for k={required_neighbor_count} "
        "and supporting comments for T0. Lower THRESHOLD_SAMPLE_AGENTS "
        "or use more target stances if this thread is too small."
    )

threshold_analyses = []

threshold_common_config = {
    "event": EVENT_NAME,
    "thread_id": THREAD_ID,
    "seed": SEED,
    "repetitions": REPETITIONS,
    "sample_size_per_repetition": THRESHOLD_SAMPLE_AGENTS,
    "sampled_agent_ids_by_repetition": sampled_threshold_agents_for_json,
    "target_stances": THRESHOLD_TARGET_STANCES,
    "temperature": TEMPERATURE,
}


for condition_index, k in enumerate(
    THRESHOLD_COUNTS,
    start=1,
):
    support_count = max(
        0,
        THRESHOLD_TOTAL_NEIGHBORS - k,
    )

    print("\n" + "=" * 72)

    print(
        f"Running threshold condition "
        f"{condition_index}/{len(THRESHOLD_COUNTS)}: "
        f"T{k}"
    )

    print(
        f"Opposing neighbors: {k} | "
        f"Supporting neighbors: {support_count} | "
        f"Agents per repetition: {THRESHOLD_SAMPLE_AGENTS} | "
        f"Repetitions: {REPETITIONS} | "
        f"Seed: {SEED}"
    )

    print("=" * 72)

    results = (
        exp.run_comment_threshold_condition(
            condition_name=(
                f"T{k}_real_comment_graph"
            ),
            agent_ids=sampled_threshold_agents_by_rep,
            opinions=INITIAL_OPINIONS,
            topic=TOPIC,
            agent_names=AGENT_NAMES,
            agent_texts=AGENT_TEXTS,
            neighbor_rankings=(
                threshold_neighbor_rankings
            ),
            neighbor_count=k,
            total_neighbor_count=THRESHOLD_TOTAL_NEIGHBORS,
            repetitions=REPETITIONS,
            seed=SEED,
            temperature=TEMPERATURE,
        )
    )

    analysis = (
        exp.analyze_user_threshold_condition(
            f"T{k}: {k} Opposing Comment Neighbors",
            results,
        )
    )

    exp.print_analysis(analysis)

    print(
        "  Opposite shift:      "
        f"{analysis['opposite_shift_count']}/"
        f"{analysis['polarized_n']} "
        f"({analysis['opposite_shift_rate']:.3f})"
    )

    print(
        "  Opposite final side: "
        f"{analysis['opposite_final_count']}/"
        f"{analysis['polarized_n']} "
        f"({analysis['opposite_final_rate']:.3f})"
    )

    print(
        "  Unknown changed rate:"
        f" {analysis['neutral_agent_changed_rate']:.3f}"
    )

    print(
        "  Avg real neighbors: "
        f"{analysis['avg_real_neighbor_count']:.3f}"
    )

    print(
        "  Avg random fallback:"
        f" {analysis['avg_random_neighbor_count']:.3f}"
    )

    print(
        "  Avg opposite neighbors:"
        f" {analysis['avg_opposite_neighbor_count']:.3f}"
    )

    print(
        "  Avg support neighbors: "
        f"{analysis['avg_support_neighbor_count']:.3f}"
    )

    condition_label = f"T{k}"
    condition_config = {
        **threshold_common_config,
        "condition": condition_label,
        "opposing_neighbors": k,
        "supporting_neighbors": support_count,
        "total_neighbors": THRESHOLD_TOTAL_NEIGHBORS,
    }

    llm_output_path = exp.build_threshold_seed_output_file_path(
        LLM_OUTPUT_DIR,
        event_name=EVENT_NAME,
        thread_id=THREAD_ID,
        model=MODEL,
        condition=condition_label,
        seed=SEED,
    )
    exp.save_llm_comment_score_outputs(
        llm_output_path,
        selected_thread=selected_thread,
        topic=TOPIC,
        model=MODEL,
        condition=condition_label,
        results=results,
        config=condition_config,
        timestamp=timestamp,
    )


    exp.print_generated_comment_samples(
        results,
        limit=5,
    )

    all_results[condition_label] = results
    all_analyses[condition_label] = analysis

    threshold_analyses.append(
        (condition_label, analysis)
    )

    time.sleep(SLEEP_BETWEEN)


# ============================================================
# Threshold summary
# ============================================================

print("\n" + "=" * 90)
print("Threshold experiment summary")
print("=" * 90)

print(
    f"{'Condition':<12}"
    f"{'Valid':>8}"
    f"{'Changed':>10}"
    f"{'ChangeRate':>12}"
    f"{'OppShift':>12}"
    f"{'AvgChange':>12}"
)

print("-" * 90)

for condition_label, analysis in threshold_analyses:
    print(
        f"{condition_label:<12}"
        f"{analysis['n']:>8}"
        f"{analysis['changed']:>10}"
        f"{analysis['changed_rate']:>12.3f}"
        f"{analysis['opposite_shift_rate']:>12.3f}"
        f"{analysis['avg_abs_change']:>12.3f}"
    )

threshold_agent_change_results = []
threshold_condition_labels = {
    condition_label
    for condition_label, _ in threshold_analyses
}
for condition_label, condition_results in all_results.items():
    if condition_label in threshold_condition_labels:
        threshold_agent_change_results.extend(condition_results)

threshold_agent_sequence_path = exp.build_threshold_agent_stance_sequence_output_file_path(
    LLM_OUTPUT_DIR,
    event_name=EVENT_NAME,
    thread_id=THREAD_ID,
    seed=SEED,
)

threshold_condition_order = [
    condition_label
    for condition_label, _ in threshold_analyses
]

exp.save_threshold_agent_stance_sequences(
    threshold_agent_sequence_path,
    selected_thread=selected_thread,
    topic=TOPIC,
    model=MODEL,
    seed=SEED,
    results=threshold_agent_change_results,
    conditions=threshold_condition_order,
    config={
        **threshold_common_config,
        "conditions": threshold_condition_order,
    },
    timestamp=timestamp,
)

print(f"Threshold agent stance sequences saved to: {threshold_agent_sequence_path}")


[Threshold] Candidate agents: 30
[Threshold] Valid agents: 30
[Threshold] Sampled unique agents: 20
[Threshold] Agents per repetition: 20
[Threshold] Dropped agents: 0

Running threshold condition 1/7: T0
Opposing neighbors: 0 | Supporting neighbors: 6 | Agents per repetition: 20 | Repetitions: 3 | Seed: 42

[T0_real_comment_graph] Starting 60 tasks: 20 unique agents x 3 repetitions
[T0_real_comment_graph] Task progress: 60/60 (100.0%) | valid=60 | failed=0
[T0_real_comment_graph] Finished | tasks=60 | valid=60 | failed=0

Condition: T0: 0 Opposing Comment Neighbors
  Valid calls:          60
  Changed:              7/60 (0.117)
  Flip rate metric:     0.117
  Keep current rate:    0.883
  Support final rate:   0.583
  Oppose final rate:    0.417
  Distribution:         {'oppose': 25, 'support': 35}
  Transition diagnostics:
    Polarity flips:     7/7 (1.000)
    Transition counts:
      oppose->support: 3
      support->oppose: 4
  Changed agents by initial stance:
    support: 3/12 

[T4_real_comment_graph] Task progress: 60/60 (100.0%) | valid=60 | failed=0
[T4_real_comment_graph] Finished | tasks=60 | valid=60 | failed=0

Condition: T4: 4 Opposing Comment Neighbors
  Valid calls:          60
  Changed:              13/60 (0.217)
  Flip rate metric:     0.217
  Keep current rate:    0.783
  Support final rate:   0.583
  Oppose final rate:    0.417
  Distribution:         {'oppose': 25, 'support': 35}
  Transition diagnostics:
    Polarity flips:     13/13 (1.000)
    Transition counts:
      oppose->support: 6
      support->oppose: 7
  Changed agents by initial stance:
    support: 3/12 (0.250)
    oppose: 4/8 (0.500)
  Changed calls by initial stance:
    support: 7/36 (0.194)
    oppose: 6/24 (0.250)
  Tokens:               63397
  Opposite shift:      13/60 (0.217)
  Opposite final side: 13/60 (0.217)
  Unknown changed rate: 0.000
  Avg real neighbors: 4.000
  Avg random fallback: 2.000
  Avg opposite neighbors: 4.000
  Avg support neighbors: 2.000
[LLM Output

## 4. Experiment 2 — Anchoring

所有目标 Agent 都看到相同的 2 个支持邻居和 2 个否认邻居，测试其最终意见更倾向于回到中立，还是保持初始立场。


In [ ]:
# Experiment 2: Anchoring

# Use binary support/oppose initial stances.
# All target agents see the same fixed shared anchor comments.
ANCHOR_FIXED_NEIGHBOR_IDS = [3, 18, 16, 21]

anchor_neighbor_items = []
for neighbor_id in ANCHOR_FIXED_NEIGHBOR_IDS:
    stance = INITIAL_OPINIONS[neighbor_id]
    if stance == "support":
        relation = "support"
    elif stance == "oppose":
        relation = "opposite"
    else:
        relation = "unknown"

    anchor_neighbor_items.append({
        "agent_id": neighbor_id,
        "source": "fixed_shared_anchor",
        "relation": relation,
    })

anchor_support_count = sum(
    item["relation"] == "support"
    for item in anchor_neighbor_items
)
anchor_opposite_count = sum(
    item["relation"] == "opposite"
    for item in anchor_neighbor_items
)
anchor_neutral_count = sum(
    item["relation"] == "neutral"
    for item in anchor_neighbor_items
)

anchor_neighbor_ids = {
    item["agent_id"]
    for item in anchor_neighbor_items
}

anchor_agents = sorted(
    agent_id
    for agent_id, stance in INITIAL_OPINIONS.items()
    if (
        agent_id != 0
        and agent_id not in anchor_neighbor_ids
        and stance in ANCHOR_TARGET_STANCES
    )
)

print(f"[Anchoring] Target agents: {anchor_agents}")
print(
    "[Anchoring] Shared anchor neighbors:",
    [
        {
            "agent_id": item["agent_id"],
            "relation": item["relation"],
            "stance": INITIAL_OPINIONS[item["agent_id"]],
            "text": AGENT_TEXTS[item["agent_id"]],
        }
        for item in anchor_neighbor_items
    ],
)

results_anchor = exp.run_comment_anchor_condition(
    condition_name="T4c_fixed_shared_anchor_comment_then_classify",
    agent_ids=anchor_agents,
    opinions=INITIAL_OPINIONS,
    topic=TOPIC,
    agent_names=AGENT_NAMES,
    agent_texts=AGENT_TEXTS,
    neighbor_items=anchor_neighbor_items,
    repetitions=REPETITIONS,
    temperature=TEMPERATURE,
)

analysis_anchor = exp.analyze_user_threshold_condition(
    "T4c: Fixed Shared Anchor Comments",
    results_anchor,
)
exp.print_analysis(analysis_anchor)

print(
    "  Opposite shift:      "
    f"{analysis_anchor['opposite_shift_count']}/"
    f"{analysis_anchor['polarized_n']} "
    f"({analysis_anchor['opposite_shift_rate']:.3f})"
)

print(
    "  Opposite final side: "
    f"{analysis_anchor['opposite_final_count']}/"
    f"{analysis_anchor['polarized_n']} "
    f"({analysis_anchor['opposite_final_rate']:.3f})"
)

print(
    "  Unknown changed rate:"
    f" {analysis_anchor['neutral_agent_changed_rate']:.3f}"
)

print(
    "  Avg opposite neighbors:"
    f" {analysis_anchor['avg_opposite_neighbor_count']:.3f}"
)

print(
    "  Avg support neighbors: "
    f"{analysis_anchor['avg_support_neighbor_count']:.3f}"
)

anchor_output_path = exp.build_llm_output_file_path(
    LLM_OUTPUT_DIR,
    selected_thread=selected_thread,
    model=MODEL,
    condition="T4c_anchor",
)

exp.save_llm_comment_score_outputs(
    anchor_output_path,
    selected_thread=selected_thread,
    topic=TOPIC,
    model=MODEL,
    condition="T4c_anchor",
    results=results_anchor,
    config={
        "thread_id": THREAD_ID,
        "condition": "T4c_anchor",
        "target_stances": ANCHOR_TARGET_STANCES,
        "fixed_anchor_neighbor_ids": ANCHOR_FIXED_NEIGHBOR_IDS,
        "supporting_neighbors": anchor_support_count,
        "opposing_neighbors": anchor_opposite_count,
        "neutral_neighbors": anchor_neutral_count,
        "shared_anchor_neighbor_ids": sorted(anchor_neighbor_ids),
        "shared_anchor_sampling": "fixed",
        "repetitions": REPETITIONS,
        "temperature": TEMPERATURE,
    },
    timestamp=timestamp,
)

exp.print_generated_comment_samples(
    results_anchor,
    limit=5,
)

all_results["T4c_anchor"] = results_anchor
all_analyses["T4c_anchor"] = analysis_anchor

print(
    "\nInterpretation: "
    "high keep_current_rate supports anchoring; "
    "high neutral_final_rate supports averaging."
)


## 5. Experiment 3 — Format Sensitivity

使用完全相同的 Agent 与邻居信息，仅改变 Prompt A/B 的表达格式，并进行逐 Agent、逐 repetition 的配对比较。


In [ ]:
# Experiment 3: Format Sensitivity

# 保持原脚本当前行为：使用除 source 节点 0 之外的全部 Agent。
format_agents = sorted(
    agent_id
    for agent_id in INITIAL_OPINIONS
    if agent_id != 0
)
print(f"[Format] Target agents: {format_agents}")

format_lines = exp.make_controlled_neighbor_lines(
    n_support=FORMAT_SUPPORTS,
    n_deny=FORMAT_DENIES,
    n_neutral=0,
)

results_format_a = exp.run_controlled_condition(
    condition_name="C1_prompt_A_same_info",
    agent_ids=format_agents,
    opinions=INITIAL_OPINIONS,
    topic=TOPIC,
    neighbor_lines=format_lines,
    prompt_variant="A",
    repetitions=REPETITIONS,
    temperature=TEMPERATURE,
)
analysis_format_a = exp.analyze_condition(
    "C1: Same Info + Prompt A",
    results_format_a,
    direction="up",
)
exp.print_analysis(analysis_format_a)

time.sleep(SLEEP_BETWEEN)

results_format_b = exp.run_controlled_condition(
    condition_name="C3_prompt_B_same_info",
    agent_ids=format_agents,
    opinions=INITIAL_OPINIONS,
    topic=TOPIC,
    neighbor_lines=format_lines,
    prompt_variant="B",
    repetitions=REPETITIONS,
    temperature=TEMPERATURE,
)
analysis_format_b = exp.analyze_condition(
    "C3: Same Info + Prompt B",
    results_format_b,
    direction="up",
)
exp.print_analysis(analysis_format_b)

comparison_format = exp.compare_pairwise(
    "C1 Prompt A vs C3 Prompt B",
    results_format_a,
    results_format_b,
)
exp.print_pairwise(comparison_format)

all_results["C1_format_A"] = results_format_a
all_results["C3_format_B"] = results_format_b
all_analyses["C1_format_A"] = analysis_format_a
all_analyses["C3_format_B"] = analysis_format_b
pairwise["format_C1_vs_C3"] = comparison_format


## 6. Experiment 4 — Label Override

在真实 PHEME 图上选择外围节点，对比“不提供角色标签”和“错误标记为 central hub”两种条件。


In [ ]:
# Experiment 4: Label Override

label_agents = sorted(peripheral_nodes[:TARGET_AGENTS])
print(f"[Label Override] True peripheral nodes: {label_agents}")

results_label_base = exp.run_real_graph_condition(
    condition_name="B7_base_peripheral_no_label_real_graph",
    agent_ids=label_agents,
    opinions=INITIAL_OPINIONS,
    graph=GRAPH,
    topic=TOPIC,
    prompt_variant="A",
    role_labels=None,
    repetitions=REPETITIONS,
    temperature=TEMPERATURE,
)
analysis_label_base = exp.analyze_condition(
    "B7-base: True Peripheral + No Label",
    results_label_base,
)
exp.print_analysis(analysis_label_base)

time.sleep(SLEEP_BETWEEN)

hub_mislabels = {
    agent_id: "central hub"
    for agent_id in label_agents
}
results_label_hub = exp.run_real_graph_condition(
    condition_name="B7_peripheral_mislabeled_as_hub_real_graph",
    agent_ids=label_agents,
    opinions=INITIAL_OPINIONS,
    graph=GRAPH,
    topic=TOPIC,
    prompt_variant="A",
    role_labels=hub_mislabels,
    repetitions=REPETITIONS,
    temperature=TEMPERATURE,
)
analysis_label_hub = exp.analyze_condition(
    "B7: True Peripheral Mislabeled as Central Hub",
    results_label_hub,
)
exp.print_analysis(analysis_label_hub)

comparison_label = exp.compare_pairwise(
    "B7-base No Label vs B7 Mislabel Hub",
    results_label_base,
    results_label_hub,
)
exp.print_pairwise(comparison_label)

all_results["B7_base"] = results_label_base
all_results["B7_mislabel_hub"] = results_label_hub
all_analyses["B7_base"] = analysis_label_base
all_analyses["B7_mislabel_hub"] = analysis_label_hub
pairwise["label_B7_base_vs_mislabel"] = comparison_label


## 7. 汇总并保存全部实验结果

In [ ]:
print("\n" + "#" * 96)
print("SUMMARY TABLE")
print("#" * 96)
print(
    f"{'Condition':<48} "
    f"{'Changed':>10} "
    f"{'Flip':>8} "
    f"{'Keep':>8} "
    f"{'Support':>8} "
    f"{'Oppose':>8}"
)
print("-" * 96)

for key, analysis in all_analyses.items():
    print(
        f"{analysis['name']:<48} "
        f"{analysis['changed']}/{analysis['n']:<6} "
        f"{analysis['avg_abs_change']:>8.3f} "
        f"{analysis['keep_rate']:>8.3f} "
        f"{analysis['support_rate']:>8.3f} "
        f"{analysis.get('oppose_rate', analysis.get('deny_rate', 0.0)):>8.3f}"
    )

config = {
    "event_dir": str(EVENT_DIR),
    "thread_id": THREAD_ID,
    "min_reactions": MIN_REACTIONS,
    "max_agents": MAX_AGENTS,
    "llm_output_dir": str(LLM_OUTPUT_DIR),
    "model": MODEL,
    "stance_source": "data",
    "temperature": TEMPERATURE,
    "repetitions": REPETITIONS,
    "max_workers": MAX_WORKERS,
    "sleep_between": SLEEP_BETWEEN,
    "max_neighbors_in_prompt": MAX_NEIGHBORS_IN_PROMPT,
    "seed": SEED,
    "target_agents": TARGET_AGENTS,
    "threshold_sample_agents": THRESHOLD_SAMPLE_AGENTS,
    "threshold_target_stances": THRESHOLD_TARGET_STANCES,
    "threshold_counts": THRESHOLD_COUNTS,
    "threshold_total_neighbors": THRESHOLD_TOTAL_NEIGHBORS,
    "threshold_support_neighbors": THRESHOLD_SUPPORT_NEIGHBORS,
    "anchor_target_stances": ANCHOR_TARGET_STANCES,
    "anchor_support_neighbors": ANCHOR_SUPPORT_NEIGHBORS,
    "anchor_opposite_neighbors": ANCHOR_OPPOSITE_NEIGHBORS,
    "format_target_stances": FORMAT_TARGET_STANCES,
    "format_supports": FORMAT_SUPPORTS,
    "format_denies": FORMAT_DENIES,
    "role_frac": ROLE_FRAC,
}

graph_summary = {
    "nodes": len(GRAPH),
    "edges": len(exp.get_undirected_edges(GRAPH)),
    "degrees": degrees,
    "initial_distribution": initial_distribution,
    "hub_nodes": hub_nodes,
    "peripheral_nodes": peripheral_nodes,
}

output_path = OUTPUT_DIR / f"four_effects_results_{timestamp}.json"
saved_path = exp.save_experiment_bundle(
    output_path,
    config=config,
    selected_thread=selected_thread,
    topic=TOPIC,
    graph_summary=graph_summary,
    analyses=all_analyses,
    pairwise=pairwise,
    results=all_results,
    timestamp=timestamp,
)

print(f"\nFull results saved to: {saved_path}")
print(f"Total API calls used: {exp.call_count}")
